# MLP (neural network) — PathSnap candidate classifier

Algorithm family: neural networks (a feed-forward multilayer perceptron).

This uses the same shared dataset, trip-level folds and metrics as every other model. The network is trained epoch by epoch with `pathsnap.modeling.EpochMLP`. That records real training and validation log loss after every epoch, which gives the loss curves the report asks for. It also early-stops on validation loss. The early-stopping validation set is 10% of the training trips; it never includes the CV validation fold or the student-test trips.

Outputs go to `models/mlp/`. Runtime is about 10–15 min with the settings below.

In [ ]:
import os, sys, json, time
from pathlib import Path
# find the repo root (the folder that contains config.yaml) and make `pathsnap` importable
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the pathsnap-ml-hackathon repo")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathsnap.dataset import FEATURES
from pathsnap import modeling as M
print("repo root:", ROOT)

## Settings

In [ ]:
ROWS_PER_FOLD = 80_000      # 5 folds -> ~400k training rows (None = all rows, much slower)
PARAMS = dict(
    hidden_layer_sizes=(64, 32),
    alpha=1e-4,                # L2 regularisation
    learning_rate_init=1e-3,   # Adam
    batch_size=512,
    max_epochs=60,
    patience=5,                # early stopping on validation log loss
    es_trip_frac=0.1,
    seed=M.SEED,
)
OUT = ROOT / "models" / "mlp"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
tr, te, tr_pts, te_pts = M.load_data()
print(f"labelled train rows: {len(tr):,}  (positive rate {tr.label.mean():.3f}, trips {tr.trip_id.nunique()})")
print(f"rows used for FITTING: up to {ROWS_PER_FOLD or 'all'} per fold; validation always uses full folds")
print(f"student-test rows: {len(te):,}")

## Model
Loss function: binary cross-entropy (log loss) plus L2 penalty `alpha`, optimised with Adam. The features are standardised, with the scaler fitted on the training rows only. Note: sklearn's MLP does not support per-sample weights, so `trip_weight` is not used for this model.

In [ ]:
def make_model():
    return M.EpochMLP(**PARAMS)

def fit_fn(model, df):
    return model.fit_df(df)

## 5-fold group cross-validation (trip-level)

In [ ]:
cv = M.cross_validate(make_model, fit_fn, tr, tr_pts, ROWS_PER_FOLD)
cv[["fold", "top1_accuracy", "train_top1", "f1_macro", "precision_macro", "recall_macro", "log_loss", "fit_seconds"]].round(4)

In [ ]:
print("CV Top-1     : %.4f ± %.4f" % (cv.top1_accuracy.mean(), cv.top1_accuracy.std()))
print("CV macro F1  : %.4f ± %.4f" % (cv.f1_macro.mean(), cv.f1_macro.std()))
print("CV log loss  : %.4f" % cv.log_loss.mean())
print("train Top-1  : %.4f" % cv.train_top1.mean())
M.plot_cv_folds(cv, "MLP: train vs validation Top-1 per fold", OUT / "cv_folds.png")

## Final fit on all training rows → student-testing trips

In [ ]:
model, scores, test, train_s, infer_ms = M.fit_and_test(make_model, fit_fn, tr, te, te_pts, ROWS_PER_FOLD)
print(f"train time: {train_s:.1f}s   epochs run: {len(model.history_)}   best epoch: {model.best_epoch_}")
print(f"inference: {infer_ms:.3f} ms per 1000 candidate rows")
pd.Series({k: v for k, v in test.items() if k != "confusion_matrix"}).to_frame("student_test")

## Training and validation loss per epoch (final model)

In [ ]:
M.plot_loss_history(model.history_, "MLP training history", OUT / "loss_curve.png", model.best_epoch_)
model.history_.to_csv(OUT / "training_history.csv", index=False)
model.history_.round(4)

In [ ]:
M.plot_confusion(test["confusion_matrix"], "MLP — student-test (threshold 0.5)", OUT / "confusion_matrix.png")

## Save results

In [ ]:
res = M.save_run(OUT, "mlp", "Neural network", PARAMS, cv, test, train_s, infer_ms,
                 model=model, rows_per_fold=ROWS_PER_FOLD,
                 extra={"epochs_run": len(model.history_), "best_epoch": model.best_epoch_})
print(json.dumps({k: v for k, v in res.items() if k != "student_test"}, indent=1, default=str))

## Notes for the report (fill in from YOUR results above)
- Why an MLP suits PathSnap, and its strengths and limitations as observed.
- Hyperparameters and initial values: see `PARAMS`. Loss: binary cross-entropy + L2, optimiser Adam.
- Over- or underfitting: read it from the loss curve (where the validation loss stops falling) and from the gap between train and validation Top-1.